In [ ]:
import json
import logging
import os
import sys

from pathlib import Path

basepath = Path(
    r"directory\aggregated_event_data"
)

sys.path.append(os.path.join(*basepath.parts))

from assembly_simulation import simulate
from numpy import isclose
from pathlib import Path

logging.basicConfig(level=logging.WARNING, force=True)
logger = logging.getLogger()

from aggregated_traces.utils.construct_ekg import (
    check_quantities,
    generate_networkx_di_graph,
    insert_DF_DP,
    insert_fractions,
    load_rdf_graph,
)
from aggregated_traces.utils.ekg_analysis import (
    compute_aggregation_uniformity,
    compute_aggregation_average_kl_trace_graph,
    compute_trace_probabilities,
    compute_number_of_merges_in_trace_graph,
    compute_number_of_steps_at_aggregations,
)

In [ ]:
from rdflib import Graph, URIRef, Variable
from typing import List


def select_packing_units_with_quality_issue(
    graph: Graph, threshold: float
) -> List[URIRef]:
    """
    Find packing unit(s) with average quality below a certain threshold.
    """

    query_lowest_quality = """
    PREFIX : <http://example.org/def/ekg/aggregated_traces/>
    SELECT DISTINCT
        ?packing_unit
        (sum(?device_quality)/count(?device_quality) as ?average_quality)
    WHERE {
        [] :bizStep "packing" ;
            :parentEntity ?packing_unit ;
            :device/:quality ?device_quality .

        ?packing_unit a :PackingUnit .
    }
    GROUP BY ?packing_unit
    """

    bindings = graph.query(query_lowest_quality).bindings

    return [
        b[Variable("packing_unit")]
        for b in bindings
        if b[Variable("average_quality")].toPython() < threshold
    ]


from pandas import DataFrame, Index, isna


def get_entity_record_number(df: DataFrame, entity: str) -> int:
    try:
        return Index(df["entity_target"]).get_loc(entity) + 1
    except KeyError:
        return None


def compute_trace_results(event_data_file: str) -> str | None:
    # Parse data to RDF graph
    ekg_graph_rdf = load_rdf_graph(event_data_file)

    # Insert DF/DP relations
    ekg_graph_rdf = insert_DF_DP(ekg_graph_rdf)

    # Check incoming vs outgoing amount
    query_result_incoming_outgoing = check_quantities(ekg_graph_rdf)

    # Insert fractions on the relations
    ekg_graph_rdf = insert_fractions(ekg_graph_rdf)

    # Create NetworkX graph
    ekg_graph_nx = generate_networkx_di_graph(ekg_graph_rdf)

    # Find packing units with lowest (average) quality for which to find the root cause (entity)
    entities_source_backward = select_packing_units_with_quality_issue(
        graph=ekg_graph_rdf, threshold=quality_threshold
    )

    # Compute backward trace probabilities
    try:
        df_backward, edges_paths_backward = compute_trace_probabilities(
            rdf_trace_graph=ekg_graph_rdf,
            nx_trace_graph=ekg_graph_nx,
            source_entities=entities_source_backward,
            trace_backward=True,
        )
    except RuntimeError as e:
        logger.warning(e)
        return None

    # Compute statistics on trace graphs for source-target node pairs
    aggregation_kl = compute_aggregation_uniformity(graph=ekg_graph_rdf)
    for i in df_backward.index:
        # Compute number of merges in the trace graph
        df_backward.loc[i, "n_merges"] = compute_number_of_merges_in_trace_graph(
            trace_graph=ekg_graph_nx,
            source=df_backward.loc[i, "node_source"],
            target=df_backward.loc[i, "node_target"],
        )

        # Compute uniformity of splits and merges
        df_backward.loc[i, "split_merge_kl"] = (
            compute_aggregation_average_kl_trace_graph(
                trace_graph=ekg_graph_nx,
                aggregation_kl=aggregation_kl,
                source=df_backward.loc[i, "node_source"],
                target=df_backward.loc[i, "node_target"],
            )
        )

    # Get the number of steps executed at the aggregation events
    steps_aggregations = compute_number_of_steps_at_aggregations(
        graph=ekg_graph_rdf, events=df_backward["node_source"].unique()
    )
    df_backward["n_production_steps_aggregations"] = df_backward["node_source"].map(
        steps_aggregations
    )

    # Validate if probabilities add up to one
    if not all(
        isclose(
            df_backward.groupby(["entity_source", "product_model"])["probability"]
            .sum()
            .astype(float),
            1,
        )
    ):
        raise Warning("Sum of probabilities by production step do not add up to one!")

    df_backward["probability"] = df_backward["probability"].astype(float)
    with open(result_files.joinpath(run_file_name), "w") as f:
        json.dump(
            df_backward.to_dict(orient="records"),
            f,
        )

    return result_files.joinpath(run_file_name)

In [ ]:
import json

from collections import defaultdict
from typing import Dict, Tuple


def generate_scenario(
    output_file: str,
    n_lots: int,
    n_devices: List[int],
    steps_resources: Dict[str, Tuple[int]],
    root_cause_resource: str,
    required_material: Dict[str, str] = None,
    merge_after_steps: List[str] = [],
    # n_lots_merge: int = 2,
    split_configs: List[dict] = [],
) -> str:
    # Select lots for merge after a certain step
    merge_configs = [{"after_step": step} for step in merge_after_steps]
    merge_configs_lots = defaultdict(list)
    for config in merge_configs:
        # for i in [i for i in range(0, n_lots, n_lots_merge)]:
        for i in range(n_lots):
            merge_configs_lots[f"Lot{i}"].append(config)

    production_lots = []
    for i in range(n_lots):
        lot_id = f"Lot{i}"
        production_lot = {
            "id": lot_id,
            "steps": list(steps_resources.keys()),
            "merge": merge_configs_lots.get(lot_id, []),
            "split": split_configs,
            "n_devices": n_devices[i],
        }

        if required_material:
            production_lot["required_material"]: required_material

        production_lots.append(production_lot)

    production_resources = []
    for step, (n, mean_duration) in steps_resources.items():
        production_resources.extend(
            [
                {
                    "id": f"{step}{i}",
                    "step": step,
                    "mean_move": 0.5,
                    "mean_duration": mean_duration,
                    "mean_breakdown": 5,
                    "mean_repair": 10,
                    "process_yield": 0.5 if root_cause_resource == f"{step}{i}" else 1,
                }
                for i in range(n)
            ]
        )

    simulation_config = {
        "production_lots": production_lots,
        "production_resources": production_resources,
        "material_lot_size": 100,
        "packing_unit_size": 50,
    }

    with open(output_file, "w") as f:
        json.dump(simulation_config, f, indent=2)

    return output_file

# Process event logs

## Compute trace probabilities

In [ ]:
n_runs = 10

# Define the root cause entity
quality_threshold = 0.9

variable = "merge_after_steps"
variable_values = [
    ["WT"],
    ["WT", "DB"],
    ["WT", "DB", "WB"],
    ["WT", "DB", "WB", "Marking"],
    ["WT", "DB", "WB", "Marking", "FT"],
]

for variable_value in variable_values:
    scenario_name = f"scenarios/{variable}={'-'.join(variable_value)}"

    generate_scenario(
        output_file=scenario_name + ".json",
        n_lots=32,
        n_devices=[50] * 32,
        steps_resources={
            "WT": (2, 1),
            "DB": (2, 2),
            "WB": (4, 20),
            "Marking": (4, 1),
            "FT": (2, 1),
        },
        root_cause_resource="WB1",
        # required_material: Dict[str, str] = None,
        merge_after_steps=variable_value,
        # n_lots_merge: int = 2,
        # split_configs: List[dict] = [],
    )

    event_log_files = basepath.joinpath(f"logs/{scenario_name}/")
    if not event_log_files.exists():
        os.mkdir(event_log_files)

    result_files = Path(f"output/{scenario_name}/")
    if not result_files.exists():
        os.mkdir(result_files)

    for i in range(n_runs):
        # if i != 0:
        #     continue

        run_file_name = f"run_{i}.json"
        event_data_file = event_log_files.joinpath(run_file_name)
        simulate.main(
            config_file=scenario_name + ".json",
            runtime=1000,
            output_event_log_file=event_data_file,
        )

        print(event_data_file)

        compute_trace_results(event_data_file)

In [ ]:
ekg_graph_rdf.query("""
SELECT DISTINCT *
WHERE {
    [] :fraction ?location
}
""").bindings

## Compute steps to find root cause (entity)

In [ ]:
def compute_n_steps(probability_dicts) -> DataFrame:
    group_key = "product_model"

    results = []
    for event_data_file, probability_dict in probability_dicts.items():
        df_trace = DataFrame(probability_dict)

        for group_value in df_trace[group_key].unique():
            # Aggregate to get probabilities for target entities
            df_trace_grouped = (
                df_trace[df_trace[group_key] == group_value]
                .groupby(["entity_source", "entity_target"])
                .agg(
                    {
                        "probability": "sum",
                        "n_merges": "sum",
                        "split_merge_kl": "mean",
                        "n_production_steps_aggregations": list,
                    }
                )
            )
            df_trace_grouped.reset_index(inplace=True)

            # Remove lists with NaN value
            df_trace_grouped["n_production_steps_aggregations"] = df_trace_grouped[
                "n_production_steps_aggregations"
            ].apply(lambda x: [] if any(not isinstance(i, list) for i in x) else x)

            entities_source = df_trace_grouped["entity_source"].unique()
            for entity_source in entities_source:
                df_selected = df_trace_grouped[
                    df_trace_grouped["entity_source"] == entity_source
                ]

                # Shuffle/sample DataFrame for random order of inspection
                n_steps_random = get_entity_record_number(
                    df_selected.sample(frac=1), root_cause_entity
                )

                # Sort values for informed order of inspection
                n_steps_sorted = get_entity_record_number(
                    df_selected.sort_values("probability", ascending=False),
                    root_cause_entity,
                )

                results.append(
                    {
                        "file": event_data_file,
                        group_key: group_value,
                        "probabilities": df_selected.to_dict(orient="records"),
                        "n_steps_random": n_steps_random,
                        "n_steps_sorted": n_steps_sorted,
                    }
                )

    return DataFrame(results)

In [ ]:
scenario_names = [
    "scenarios/merge_after_steps=WT",
    "scenarios/merge_after_steps=WT-DB",
    "scenarios/merge_after_steps=WT-DB-WB",
    "scenarios/merge_after_steps=WT-DB-WB-Marking",
    "scenarios/merge_after_steps=WT-DB-WB-Marking-FT",
]

scenario_dict = {}
for scenario_name in scenario_names:
    print(scenario_name)

    with open(scenario_name + ".json") as f:
        config = json.load(f)
    lower_yield_resources = [
        resource["id"]
        for resource in config["production_resources"]
        if resource["process_yield"] < 1
    ]
    if len(lower_yield_resources) > 1:
        print("Multiple resources with yield < 1")

    root_cause_entity = (
        f"http://example.org/id/ekg/aggregated_traces/{lower_yield_resources[0]}"
    )

    probability_dicts = {}
    for result_file in Path(f"output/{scenario_name}/").glob("*.json"):
        with open(result_file) as f:
            probability_dicts[result_file] = json.load(f)

        scenario_dict[scenario_name] = compute_n_steps(probability_dicts)

In [ ]:
from pandas import concat

df_combined = DataFrame()
for k, v in scenario_dict.items():
    v["scenario_name"] = k
    df_combined = concat([df_combined, v])

df_combined["n_steps_diff"] = (
    df_combined["n_steps_random"] - df_combined["n_steps_sorted"]
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from pandas import isna, set_option
from scipy.stats import pearsonr

set_option("display.max_colwidth", None)

In [ ]:
# If "n_steps_random" is NaN, the root cause entity does not occur in the trace
df_combined[~isna(df_combined["n_steps_random"])].describe()

In [ ]:
df_combined.groupby("scenario_name")[
    ["n_steps_diff", "n_steps_random", "n_steps_sorted"]
].describe().T

In [ ]:
fig, ax = plt.subplots(1, 1)

ax = sns.boxplot(
    data=df_combined.dropna(subset=["n_steps_diff"]),
    x="scenario_name",
    y="n_steps_diff",
)

ax.tick_params(axis="x", rotation=90)

In [ ]:
from aggregated_traces.utils.visualization import generate_graph_visualization

visualize_ekg_graph_rdf = load_rdf_graph(
    basepath.joinpath("logs/scenarios/merge_after_steps=WT-DB/run_0.json")
)

# Insert DF/DP relations
visualize_ekg_graph_rdf = insert_DF_DP(visualize_ekg_graph_rdf)

# Insert fractions on the relations
visualize_ekg_graph_rdf = insert_fractions(visualize_ekg_graph_rdf)

# Create NetworkX graph
visualize_ekg_graph_nx = generate_networkx_di_graph(visualize_ekg_graph_rdf)

generate_graph_visualization(visualize_ekg_graph_nx, base_figure_path="output/check")